In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_07_contacts_attributes
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_07_contacts_attributes
# Spec Ref  : §1.9 contacts_attributes (contact_id, is_paying, is_excluded, mrr)
# Purpose   : Create one attributes row for EVERY contact.
# Run after : map_01 (contacts_all)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
print(f"=== Map 07: contacts_attributes ===  customer={customer_id}")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"

import sys, os
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
if os.path.abspath(project_root) not in sys.path:
    sys.path.append(os.path.abspath(project_root))

from utils.pipeline_metrics import PipelineMetrics
pm = PipelineMetrics(
    spark          = spark,
    parent_run_id  = parent_run_id,
    job_name       = "02b_map_07_contacts_attributes",
    task_key       = "run_job_D_silver_map",
    source_system  = source_system,
    load_type      = load_type,
    customer_id    = customer_id,
    object_name    = object_name,
)
pm.init()

initialize_audit_tables()

In [0]:
# CELL 3
create_map_table(
    f"{sv}.contacts_attributes",
    """
        contact_id   STRING NOT NULL,
        tenant       BIGINT,
        is_paying    BOOLEAN,
        is_excluded  BOOLEAN,
        mrr          DOUBLE
    """,
    cluster_by="contact_id"
)

In [0]:
# CELL 4
try:
    # Safe drop in case target exists as a VIEW
    safe_drop_view(f"{sv}.contacts_attributes")

    spark.sql(f"""
        CREATE OR REPLACE TABLE {sv}.contacts_attributes
        USING DELTA
        CLUSTER BY (contact_id)
        {DELTA_TBLPROPS_MAP}
        AS
        SELECT
            id      AS contact_id,
            tenant,
            false   AS is_paying,
            false   AS is_excluded,
            0.0     AS mrr
        FROM {sv}.contacts_all
    """)

    n = cnt(f"{sv}.contacts_attributes")
    print(f"  contacts_attributes: {n:,} rows")
except Exception as e:
    print(f"❌ contacts_attributes build failed: {e}")
    log_audit(
        job_name       = "02b_map_07_contacts_attributes",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
try:
    total_rows = spark.table(f"{sv}.contacts_attributes").count()
    pm.save(status="SUCCESS", rows_processed=total_rows)
except Exception as e:
    print(f"❌ Metrics save failed: {e}")
    log_audit(
        job_name       = "02b_map_07_contacts_attributes",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise